# 14 — Batch Correction in Perturb-seq

**Tools:** `harmonypy`, `scvi-tools`  
**Dataset:** Norman 2019 CRISPRa (with simulated second batch)  
**Papers:** [Korsunsky et al. 2019, Nature Methods (Harmony)](https://doi.org/10.1038/s41592-019-0619-0) · [Lopez et al. 2018, Nature Methods (scVI)](https://doi.org/10.1038/s41592-018-0229-2)

**Prerequisites:** Run notebooks 03 and 07. Requires `../data/norman_assigned.h5ad`.

---

## Why batch correction matters in Perturb-seq

Multi-batch Perturb-seq is the norm at scale:
- Large screens split across multiple 10x runs or sequencing lanes
- Experiments done at different time points
- Combined datasets (e.g., Norman + Replogle)

Without correction, batch effects dominate the UMAP — cells cluster by sequencing run, not biology. This inflates false positives in DE and makes cross-perturbation comparison unreliable.

## The critical Perturb-seq-specific complication

> **Batch correction must preserve perturbation signal, not remove it.**

Standard batch correction algorithms assume you want to align *all* cells. But in Perturb-seq, perturbations ARE the signal. Two approaches:

1. **Regress out batch before perturbation analysis:** Correct on NTC cells only, project perturbed cells into the corrected space
2. **Include perturbation as a covariate in the batch model:** scVI's `batch_key` + `continuous_covariate_keys` can handle this
3. **Correct only for UMAP/visualization:** Run Harmony on NTC-anchored PCA, keep raw counts for DE

## Tools overview

| Tool | Method | Speed | GPU | Key advantage |
|------|--------|-------|-----|---------------|
| **Harmony** | Iterative soft clustering | Fast | No | Simple, no training; works on existing PCA |
| **scVI** | Variational autoencoder | Medium | Optional | Full probabilistic model; handles count noise |
| **scANVI** | Semi-supervised VAE | Slower | Optional | Leverages cell type labels |
| **scPoli** | Online learning VAE | Fast | Optional | Reference-based; good for large atlas integration |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import harmonypy as hm
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'harmonypy {hm.__version__}  |  scvi {scvi.__version__}')

## 1. Simulate a Multi-Batch Perturb-seq Experiment

The Norman dataset is single-batch. We simulate a second batch by introducing a technical shift (scaled gene expression offset + downsampling) to half the cells.

In [ ]:
adata = sc.read_h5ad('../data/norman_assigned.h5ad')
print(adata)

In [ ]:
# Subset to NTC + top 5 perturbations for speed
keep_genes = ['NTC', 'CBX2', 'SET', 'RUNX1', 'CEBPA', 'KLF1']
adata = adata[adata.obs['gene'].isin(keep_genes)].copy()
print(f'Subset: {adata.n_obs} cells')

# Save raw counts before any normalization
if 'counts' not in adata.layers:
    adata.layers['counts'] = adata.X.copy()

In [ ]:
# Simulate batch 2: take 40% of cells and introduce a technical shift
np.random.seed(42)
batch2_mask = np.random.rand(adata.n_obs) < 0.4
adata.obs['batch'] = np.where(batch2_mask, 'batch2', 'batch1')

# Technical shift: batch2 has ~20% fewer counts per cell (simulates capture efficiency diff)
if hasattr(adata.layers['counts'], 'toarray'):
    counts = adata.layers['counts'].toarray()
else:
    counts = np.array(adata.layers['counts'])

# Scale batch2 cells down by ~20%
counts[batch2_mask] = np.random.binomial(
    n=counts[batch2_mask].astype(int),
    p=0.80
)
import scipy.sparse as sp
adata.layers['counts'] = sp.csr_matrix(counts)

print('Batch distribution:')
print(adata.obs[['batch', 'gene']].value_counts().unstack().fillna(0).astype(int))

In [ ]:
# Normalize for visualization (NTC-anchored, as in notebook 07)
ntc_mask = adata.obs['gene'] == 'NTC'
adata_ntc = adata[ntc_mask].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Fit HVGs on NTC cells only
sc.pp.highly_variable_genes(adata[ntc_mask], n_top_genes=2000)
adata.var['highly_variable'] = adata[ntc_mask].var['highly_variable']
adata_hvg = adata[:, adata.var.highly_variable].copy()

# NTC-anchored PCA
from sklearn.decomposition import PCA
scaler_data = adata_hvg[ntc_mask].X
if hasattr(scaler_data, 'toarray'):
    scaler_data = scaler_data.toarray()
pca = PCA(n_components=30, random_state=42)
pca.fit(scaler_data)
if hasattr(adata_hvg.X, 'toarray'):
    adata_hvg.obsm['X_pca'] = pca.transform(adata_hvg.X.toarray())
else:
    adata_hvg.obsm['X_pca'] = pca.transform(adata_hvg.X)

sc.pp.neighbors(adata_hvg, use_rep='X_pca')
sc.tl.umap(adata_hvg)

# Carry results back
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.obsm['X_umap'] = adata_hvg.obsm['X_umap']

In [ ]:
# Before correction: UMAP should show batch separation
sc.pl.umap(adata, color=['batch', 'gene'],
           title=['Before correction: by batch', 'Before correction: by perturbation'])

## 2. Harmony: Fast PCA-Space Correction

In [ ]:
# Harmony operates directly on the PCA embedding
# It iteratively clusters cells and adjusts PCA coords to remove batch effects
# Key parameters:
#   theta: diversity penalty (higher = more aggressive correction)
#   nclust: number of clusters (default: min(100, n_cells/30))
ho = hm.run_harmony(
    adata.obsm['X_pca'],
    adata.obs,
    vars_use='batch',
    theta=2.0,          # 2 = standard; increase if batches are very different
    random_state=42,
    verbose=True,
)
adata.obsm['X_harmony'] = ho.Z_corr.T
print('Harmony corrected embedding:', adata.obsm['X_harmony'].shape)

In [ ]:
# Build new UMAP from Harmony-corrected PCA
sc.pp.neighbors(adata, use_rep='X_harmony')
sc.tl.umap(adata)
adata.obsm['X_umap_harmony'] = adata.obsm['X_umap'].copy()

sc.pl.umap(adata, color=['batch', 'gene'],
           title=['Harmony: by batch', 'Harmony: by perturbation'])

## 3. scVI: Deep Generative Model Correction

See T01 for the full scVI tutorial. Here we apply it in the Perturb-seq context, **explicitly preserving perturbation identity** by including it as a covariate.

In [ ]:
# scVI setup
# batch_key: batch to correct for
# categorical_covariate_keys: perturbation identity (preserve this signal!)
scvi.model.SCVI.setup_anndata(
    adata,
    layer='counts',
    batch_key='batch',
    categorical_covariate_keys=['gene'],   # tells scVI this is a covariate, not noise
)

model_scvi = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_latent=20,
    gene_likelihood='nb',
)
model_scvi.train(max_epochs=200, early_stopping=True)

In [ ]:
adata.obsm['X_scVI'] = model_scvi.get_latent_representation()

sc.pp.neighbors(adata, use_rep='X_scVI')
sc.tl.umap(adata)
adata.obsm['X_umap_scvi'] = adata.obsm['X_umap'].copy()

sc.pl.umap(adata, color=['batch', 'gene'],
           title=['scVI: by batch', 'scVI: by perturbation'])

## 4. Batch Mixing Assessment: kBET

kBET (k-nearest neighbour Batch Effect Test) quantifies how well-mixed cells are after correction. A good correction has high acceptance rate (kBET ≈ 0 = no mixing; ≈ 1 = perfect mixing).

In [ ]:
# Simple batch mixing metric: for each cell, what fraction of its k neighbors
# come from the other batch? (Expected = batch2_fraction)
from sklearn.neighbors import NearestNeighbors

def local_batch_mixing(embedding, batch_labels, k=30):
    """Returns per-cell neighbor batch mixing score."""
    nn = NearestNeighbors(n_neighbors=k, algorithm='ball_tree').fit(embedding)
    _, indices = nn.kneighbors(embedding)
    batch2_global = (batch_labels == 'batch2').mean()
    
    scores = []
    for i, idx in enumerate(indices):
        batch2_local = (batch_labels[idx] == 'batch2').mean()
        scores.append(batch2_local)
    return np.array(scores)

batch_labels = adata.obs['batch'].values

mixing = {
    'Before (PCA)': local_batch_mixing(adata.obsm['X_pca'], batch_labels),
    'Harmony':      local_batch_mixing(adata.obsm['X_harmony'], batch_labels),
    'scVI':         local_batch_mixing(adata.obsm['X_scVI'], batch_labels),
}

fig, ax = plt.subplots(figsize=(7, 4))
for label, scores in mixing.items():
    ax.hist(scores, bins=30, alpha=0.5, label=f'{label} (mean={scores.mean():.2f})')
ax.axvline(0.4, color='black', linestyle='--', label='Expected (40% batch2)')
ax.set_xlabel('Fraction of batch2 in 30-NN')
ax.set_title('Local batch mixing (higher = better corrected)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Critical Decision: When to Apply Batch Correction

For Perturb-seq the order matters:

In [ ]:
# Recommended workflow for multi-batch Perturb-seq:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  raw counts per batch                                                   │
# │       ↓                                                                 │
# │  Per-batch QC (filter cells, genes)                                     │
# │       ↓                                                                 │
# │  Concatenate batches  (ad.concat([b1, b2], keys=['batch1','batch2']))   │
# │       ↓                                                                 │
# │  NTC-anchored normalization + PCA  (notebook 07)                        │
# │       ↓                                                                 │
# │  Harmony OR scVI (batch_key='batch', categorical_covariates=['gene'])   │
# │       ↓                                                                 │
# │  UMAP + Leiden on corrected embedding                                   │
# │       ↓                                                                 │
# │  Mixscape (notebook 09) — uses raw perturbation signature, not PCA     │
# │       ↓                                                                 │
# │  Pseudobulk DE (notebook 08) — uses raw counts, not corrected          │
# └─────────────────────────────────────────────────────────────────────────┘
#
# KEY RULE: Batch correction affects the EMBEDDING used for visualization/Milo.
#           DE always uses raw counts (pseudobulk).
#           Mixscape uses the perturbation signature (cell - NTC_mean),
#           not the corrected embedding directly.

print('Recommended batch correction placement:')
print('  Embedding/UMAP/Milo: use Harmony or scVI latent space')
print('  DE (pseudobulk): always use raw counts in design matrix')
print('  Mixscape: compute perturbation signature on NTC-batch-matched cells')

---
## Summary

| Method | Code | Best for |
|--------|------|----------|
| Harmony | `hm.run_harmony(X_pca, obs, vars_use='batch')` → `X_harmony` | Fast, exploratory; good default |
| scVI | `SCVI.setup_anndata(adata, layer='counts', batch_key='batch')` | Full probabilistic model, large datasets |
| scANVI | Initialize from scVI, add `labels_key` | When cell type labels are available |

**Perturb-seq-specific rule:** Always include perturbation identity (`categorical_covariate_keys=['gene']`) when running scVI for batch correction, so the model treats it as signal rather than removing it.